# Data Mexico

Ya tenemos la información de la producción agricola a nivel nacional y municipal. Lo que haremos ahora es extraer información de la exportación e importación de los mismos productos. Esto para ver si la producción nacional es lo suficiente para 

* Tener buena ganacia mediante exportación.
* Minimizar la importación. (Idealmente no deberiamos importar nuestros mejores productos agricolas)

Para ello ocupamos cierta información de la Secretaria de Economia.
* Balanza comercial. (Revisar el deficit de cada producto)
* Cantidad importada en Toneladas
* Cantidad exportada en Toneladas
* Precio de Venta y Precio de Compra. (A que precio vendemos y cuanto nos cuesta importar los productos agricolas)

Esperamos obtener toda esta información mediante DataMexico. Una herramienta desarrollada por la propia Secretaria de Economia para el acceso de sus bases de datos. Aunque posiblemente usaremos información del INEGI para información faltante y/o calidad de Datos.

Esto porque en México la información de "mejor calidad" es la otorgada por el INEGI.

## Diccionario de Datos

Al igual que en el DGSIAP. Lo primero es obtener el diccionario de datos respectivo para poder utilizar sus conjuntos de datos de forma adecuada.

Como DataMexico no tiene porque usar la misma estructura que el DGSIAP debemos analizar su API para poder encontrarlo. En este caso, gracias a la organización de DataMexico, todos los diccionarios estan en un solo archivo que es facilmente obtenible.

In [1]:
import yarl
import requests
import json
from scripts import download as d
import pandas as pd
from scripts.download import DATA_PATH

datadict_url = "https://www.economia.gob.mx/datamexico/files/diccionario.xlsx"
tigie_to_scian = "https://www.inegi.org.mx/contenidos/app/tigie/tabla-correlacion-actualizacion-2025.xlsx?v=0.0.1" 
DATAMEXICO_DICT = yarl.URL(datadict_url)
TIGIE_CATALOG = yarl.URL(tigie_to_scian)

filename_1 = datadict_url.split("/")[-1]
filename_2 = tigie_to_scian.split("/")[-1].split("?")[0]

d.download(DATAMEXICO_DICT, filename_1)
d.download(TIGIE_CATALOG, filename_2)


El archivo ya fue descargado. Se encuentra en /data/external/diccionario.xlsx
El archivo ya fue descargado. Se encuentra en /data/external/tabla-correlacion-actualizacion-2025.xlsx


### TIGIE
El TIGIE o Tarifa de la Ley de los Impuestos Generales de Importación y de Exportación es el documento oficial en México que establece los aranceles y las normas para clasificar todas las mercancías que entran o salen del país. 

#### Catalogo
Ellos poseen un catalogo, que descargamos como  "tabla-correlacion-actualizacion-2025.xlsx" donde etiquetan cada producto que Mexico somete a exportación e importación.

In [2]:
tigie_scian = pd.read_excel(DATA_PATH / filename_2, skiprows=2)
print(tigie_scian.head(n=3))
# Vemos las columnas para saber donde hacer el corte
print(tigie_scian.columns) 

tigie_scian = tigie_scian.drop(axis=0, columns=['Unnamed: 2'])
tigie_scian.head(n=5)

      TIGIE 2022                        Unnamed: 1  Unnamed: 2  \
0         Código                            Nombre         NaN   
1  0101.21.01.00       Reproductores de raza pura.         NaN   
2  0101.29.02.00  Sin pedigree, para reproducción.         NaN   

  SCIAN México 2023               Unnamed: 4  
0            Código                   Nombre  
1            112920  Explotación de équidos   
2            112920  Explotación de équidos   
Index(['TIGIE 2022', 'Unnamed: 1', 'Unnamed: 2', 'SCIAN México 2023',
       'Unnamed: 4'],
      dtype='str')


,TIGIE 2022,Unnamed: 1,SCIAN México 2023,Unnamed: 4
0,Código,Nombre,Código,Nombre
1,0101.21.01.00,Reproductores de raza pura.,112920,Explotación de équidos
2,0101.29.02.00,"Sin pedigree, para reproducción.",112920,Explotación de équidos
3,0101.29.03.00,"Para abasto, cuando la importación la realicen...",112920,Explotación de équidos
4,0101.29.99.01,Para saltos o carreras.,112920,Explotación de équidos


Como podemos ver. El documento tambien incluye su equivalencia con el catalogo del SCIAN (Sistema de Clasificación Industrial de América del Norte), que es relativamente lo mismo pero sobre las actividades economicas. Entonces de este documento podemos obtener conexiones entre las importaciones y exportaciones del pais y la inversión extranjera.

In [3]:
base_cols = set(tigie_scian.loc[0])
print(base_cols)
suffix = list(tigie_scian.columns)[0::2]
print(suffix)
new_cols = [f"{base}_{inst.split(" ")[0]}-{inst.split(" ")[-1]}" for inst in suffix for base in base_cols]
new_cols[-2] = f"{new_cols[-2]}_MX"
new_cols[-1] = f"{new_cols[-1]}_MX"
print(new_cols)


# Colocamos las nuevas columnas
tigie_scian.columns = new_cols
tigie_scian = tigie_scian.iloc[1:]
tigie_scian

{'Código', 'Nombre'}
['TIGIE 2022', 'SCIAN México 2023']
['Código_TIGIE-2022', 'Nombre_TIGIE-2022', 'Código_SCIAN-2023_MX', 'Nombre_SCIAN-2023_MX']


,Código_TIGIE-2022,Nombre_TIGIE-2022,Código_SCIAN-2023_MX,Nombre_SCIAN-2023_MX
1,0101.21.01.00,Reproductores de raza pura.,112920,Explotación de équidos
2,0101.29.02.00,"Sin pedigree, para reproducción.",112920,Explotación de équidos
3,0101.29.03.00,"Para abasto, cuando la importación la realicen...",112920,Explotación de équidos
4,0101.29.99.01,Para saltos o carreras.,112920,Explotación de équidos
5,0101.29.99.99,Los demás.,112920,Explotación de équidos
...,...,...,...,...
11491,9705.31.99.00,Las demás.,339999,Otras industrias manufactureras
11492,9705.39.01.00,"Objetos de interés histórico, que no hayan sid...",339999,Otras industrias manufactureras
11493,9705.39.99.00,Las demás.,339999,Otras industrias manufactureras
11494,9706.10.01.00,De más de 250 años.,339999,Otras industrias manufactureras



## Tesseract Cubes

La api de DATA MEXICO usa Data Cubes para almacenar sus datos. Lo que implica una nueva estructura de datos. Distinta al del DGSIAP que se trabajo mediante HTML Scrapping. Usando su herramienta visual VizBuilder pudimos obtener un ejemplo.

In [4]:
API_URL_EX = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Country%2CContinent%2CYear%2CIndustry+Group&limit=50&measures=Investment%2CCount"
API_EX = yarl.URL(API_URL_EX)

response = requests.get(API_EX, stream=True)
print(response.status_code)
if response.status_code == 200:
    content = response.text
    raw_data = json.loads(content)

print(raw_data)

200
{'data': [{'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 3231, 'Industry Group': 'Printing and Related Industries', 'Investment': -0.0016870000399649143, 'Count': 9}, {'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 3311, 'Industry Group': 'Iron and Steel Basic Industry', 'Investment': 0.39920100569725037, 'Count': 5}, {'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 4321, 'Industry Group': 'Wholesale Trade of Textile Products and Footwear', 'Investment': 0.012419000267982483, 'Count': 3}, {'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 4332, 'Industry Group': 'Wholesale Trade of Perfumery Products, Cosmetics and Jewelry', 'Investment': 0.0326660014

Entonces construiremos la estructura para descargar datos de Data México.

In [5]:
database = "fdi_year_industry_country"
options = ["Country", "Continent", "Year", "Industry Group"]
measures = ["Investment","Count"]


def build_data_mexico_url(data, opts, msres):
    api_sep = ","
    url = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords"
    api = yarl.URL(url)
    drilldowns = api_sep.join(opts)
    measures = api_sep.join(msres)
    link = api.with_query(cube=data, drilldowns=drilldowns, measures=measures)
    return link


def info_data_mexico_api(link):
    v1 = requests.get(link, stream=True)
    vlt = v1.text
    vjs = json.loads(vlt)
    return vjs

# link_1 = build_data_mexico_url(database, options, measures)
# print(link_1)
# h1 = info_data_mexico_api(link_1)
# print(h1)

In [7]:
q = yarl.URL("https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords")
database = "fdi_year_industry_country"
qq = q.with_query(cube=database)
options = ["Country", "Continent", "Year", "Industry Group"]
measures = ["Investment","Count"]
pars = {
    "drilldowns": options,
    "measures": measures
}

link = build_data_mexico_url(database, options, measures)
h1 = info_data_mexico_api(link)


In [8]:
from scripts.datamexico import DATA_MEXICO

def build_data_mexico_url(data, opts, msres):
    api_sep = ","
    url = DATA_MEXICO
    api = yarl.URL(url)
    drilldowns = api_sep.join(opts)
    measures = api_sep.join(msres)
    link = api.with_query(cube=data, drilldowns=drilldowns, measures=measures)
    return link


def info_data_mexico_api(link):
    v1 = requests.get(link, stream=True)
    vlt = v1.text
    vjs = json.loads(vlt)
    return vjs



database = "fdi_year_industry_country"
options = ["Country", "Continent", "Year", "Industry Group"]
measures = ["Investment","Count"]
pars = {
    "drilldowns": options,
    "measures": measures
}

link_1 = build_data_mexico_url(database, options, measures)
h1 = info_data_mexico_api(link_1)
print(h1)

{'data': [{'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 3231, 'Industry Group': 'Printing and Related Industries', 'Investment': -0.0016870000399649143, 'Count': 9}, {'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 3311, 'Industry Group': 'Iron and Steel Basic Industry', 'Investment': 0.39920100569725037, 'Count': 5}, {'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 4321, 'Industry Group': 'Wholesale Trade of Textile Products and Footwear', 'Investment': 0.012419000267982483, 'Count': 3}, {'Country ID': 'arg', 'Country': 'Argentina', 'Continent ID': 'sa', 'Continent': 'South America', 'Year': 1999, 'Industry Group ID': 4332, 'Industry Group': 'Wholesale Trade of Perfumery Products, Cosmetics and Jewelry', 'Investment': 0.03266600146889

In [13]:
pd.DataFrame(h1['data']).head(n=5)

,Country ID,Country,Continent ID,Continent,Year,Industry Group ID,Industry Group,Investment,Count
0,arg,Argentina,sa,South America,1999,3231,Printing and Related Industries,-0.001687,9
1,arg,Argentina,sa,South America,1999,3311,Iron and Steel Basic Industry,0.399201,5
2,arg,Argentina,sa,South America,1999,4321,Wholesale Trade of Textile Products and Footwear,0.012419,3
3,arg,Argentina,sa,South America,1999,4332,"Wholesale Trade of Perfumery Products, Cosmeti...",0.032666,9
4,arg,Argentina,sa,South America,1999,5312,Real Estate Agencies and Brokers,0.465271,5


In [ ]:
s = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Year%2CHS4+4+Digit%2CNation%2CProduct+Level&locale=es&measures=Trade+Value"
h3 = info_data_mexico_api(s)
vjs3 = json.loads(h3)

In [ ]:
print(vjs3.keys())

dict_keys(['data', 'source'])


In [ ]:
def data_mexico(data, opts, msres):
    api_sep = ","
    url = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords"
    api = yarl.URL(url)
    drilldowns = api_sep.join(opts)
    measures = api_sep.join(msres)
    link = api.with_query(cube=data, drilldowns=drilldowns, measures=measures)
    response = requests.get(link, timeout=30)
    print(response.status_code)
    content = response.text
    print(content)
    return json.loads(content)


q = data_mexico(database,
            ["Country", "Continent", "Year", "Industry Group"],
            ["Investment","Count"])

print(q)

200
{"data":[{"Country ID":"arg","Country":"Argentina","Continent ID":"sa","Continent":"South America","Year":1999,"Industry Group ID":3231,"Industry Group":"Printing and Related Industries","Investment":-0.0016870000399649143,"Count":9},{"Country ID":"arg","Country":"Argentina","Continent ID":"sa","Continent":"South America","Year":1999,"Industry Group ID":3311,"Industry Group":"Iron and Steel Basic Industry","Investment":0.39920100569725037,"Count":5},{"Country ID":"arg","Country":"Argentina","Continent ID":"sa","Continent":"South America","Year":1999,"Industry Group ID":4321,"Industry Group":"Wholesale Trade of Textile Products and Footwear","Investment":0.012419000267982483,"Count":3},{"Country ID":"arg","Country":"Argentina","Continent ID":"sa","Continent":"South America","Year":1999,"Industry Group ID":4332,"Industry Group":"Wholesale Trade of Perfumery Products, Cosmetics and Jewelry","Investment":0.032666001468896866,"Count":9},{"Country ID":"arg","Country":"Argentina","Contine

In [ ]:
DATA_MEXICO_API = yarl.URL("https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords")
FDI_CUBE = "cube=fdi_year_industry_country"
FDI_INDUSTRY = ["Industry+Group", "Sector", "Subsector"]
FDI_PERIOD = ["Year"]
FDI_REGION = ["Contitent", "Country"]
FDI_MEASURES = ["Investment","Count"]
DRILLDOWNS= "Industry+Group%2CYear%2CCountry"
MEASURES = "measures=Investment%2CCount"

LOCALE = "es"

FDI_SCUBE = "fdi_year_state_industry"
FDI_REGIONS = ["Nation", "State"]

FDI_I_CUBE = "fdi_4_investment_type"
FDI_TYPE = ["Investment+Type"]
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_4_investment_type&drilldowns=Investment+Type%2CState%2CNation%2CYear&locale=es&measures=Investment%2CCount

FDI_IT_CUBE = "fdi_3_country_origin"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_3_country_origin&drilldowns=Continent%2CNation%2CState%2CYear%2CCountry&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_2_state_investment&drilldowns=Year&locale=es&measures=Investment%2CCount

FDI_IT_S = "fdi_2_state_investment"

FDI_Y = "fdi_year_industry"

FDI_Y_IT_I = "fdi_year_investment_industry"

TRADE_12 = "trade_i_baci_a_12"

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=trade_i_baci_a_12&drilldowns=Year%2CExporter+Country%2CImporter+Country&locale=es&measures=Trade+Value%2CQuantity
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=trade_i_baci_a_12&drilldowns=HS6%2CYear&locale=es&measures=Trade+Value%2CQuantity


BANXICO = "banxico_trade_flow"

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=banxico_trade_flow&drilldowns=Year%2CCountry%2CFlow%2CContinent&locale=es&measures=Trade+Value


https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_9_year&drilldowns=Year&locale=es&measures=Investment%2CCount


BCMM_ECONOMY_STATE = "economy_foreign_trade_ent"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_ent&drilldowns=HS6%2CYear%2CDate+Year&locale=es&measures=Trade+Value



BCMM_ECONOMY_MUN = "economy_foreign_trade_mun"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_mun&drilldowns=HS6%2CMunicipality%2CState%2CFlow&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_mun&drilldowns=HS6%2CState%2CFlow%2CMetro+Area&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Country%2CHS6&locale=es&measures=Trade+Value


BCMM_ECONOMY_NAT = "economy_foreign_trade_nat"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Product+Level%2CYear%2CHS6&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Product+Level%2CYear%2CCountry&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Country%2CHS6%2CDate+Year&locale=es&measures=Trade+Value


INEGI_NATION = "inegi_foreign_trade_country"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=inegi_foreign_trade_country&drilldowns=Year%2CCountry%2CFlow&locale=es&measures=Trade+Value

INEGI_STATE = "inegi_foreign_trade_state"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=inegi_foreign_trade_state&drilldowns=Year%2CNation%2CState&locale=es&measures=Trade+Value


INEGI_GDP = "inegi_gdp"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=inegi_gdp&drilldowns=Year%2CSector&locale=es&measures=GDP


SNIIM = "sniim_products"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=sniim_products&drilldowns=Year%2CUnit%2CTrademark%2CNation%2CState%2CProduct+System%2CProduct+Subsystem%2CProduct%2CHS2+2+Digit&locale=es&measures=Prices




INEGI_PRODUCT = "inegi_foreign_trade_product"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=inegi_foreign_trade_product&drilldowns=Year%2CHS2+2+Digit%2CFlow&locale=es&measures=Trade+Value

TRADE_BCMM = "economy_foreign_trade_nat"
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=HS6%2CDate+Year%2CCountry&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=HS6%2CDate+Year&locale=es&measures=Trade+Value

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=HS6%2CDate+Year%2CCountry&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_fdi&drilldowns=Industry+Group%2CCountry%2CYear%2CNation%2CState%2CGeneric+Investment%2CContinent&locale=es&measures=Movement
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Date+Year%2CCountry%2CHS4+4+Digit&locale=es&measures=Trade+Value

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_foreign_trade_nat&drilldowns=Date+Year%2CCountry%2CHS2+2+Digit&locale=es&measures=Trade+Value
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=economy_fdi&drilldowns=Generic+Investment&locale=en&measures=Movement

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry&drilldowns=Industry+Group%2CYear&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_9_year&drilldowns=Year&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_9_year_investment&drilldowns=Investment+Type%2CYear&locale=es&measures=Investment%2CCount



https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_investment_industry&drilldowns=Year%2CInvestment+Type%2CIndustry+Group&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_state_industry&drilldowns=Year%2CIndustry+Group%2CNation%2CState&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_investment_industry&drilldowns=Year%2CInvestment+Type%2CIndustry+Group&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_2_state_investment&drilldowns=Year%2CInvestment+Type%2CNation%2CState&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry&drilldowns=Year%2CIndustry+Group&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry&drilldowns=Year%2CIndustry+Group&locale=es&measures=Investment%2CCount


https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_3_country_origin&drilldowns=Continent&locale=es&measures=Investment%2CCount

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Country%2CIndustry+Group%2CContinent%2CYear&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_state_industry&drilldowns=Nation%2CState%2CYear%2CIndustry+Group&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_state_industry&drilldowns=Nation%2CState%2CYear%2CIndustry+Group&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Industry+Group&limit=50&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CIndustry+Group&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Industry+Group%2CYear%2CCountry&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CSubsector&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CSector&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CIndustry+Group%2CContinent&locale=es&measures=Investment%2CCount
https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Continent%2CIndustry+Group%2CYear%2CCountry&locale=es&measures=Investment%2CCount
s = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Industry+Group&locale=es&measures=Investment%2CCount"

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Continent%2CIndustry+Group%2CYear&locale=es&measures=Investment%2CCount




SyntaxError: invalid decimal literal (4064896872.py, line 17)

In [ ]:
# Diccionario de Datos
fdi = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CIndustry+Group&measures=Investment%2CCount"
fdi = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CIndustry+Group&locale=es&measures=Investment%2CCount"

tigie_scian = "https://www.inegi.org.mx/contenidos/app/tigie/tabla-correlacion-actualizacion-2025.xlsx?v=0.0.1"
